# Phân tích chuỗi thời gian

Chạy từ trên xuống. Dùng snapshot đóng gói đến30/09/2025. Notebook09 cần chạy06 hoặc00 trước. Kiểm định không chứng minh tính dừng; SHAP không chứng minh nhân quả. Xem REPORT để đọc giả định.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## Chạy phân tích

Các bảng và hình được lưu trong reports/ và figures/.

In [ ]:
from time_series_analysis import run_analysis
result=run_analysis()
display(pd.read_csv(REPORTS_DIR / "stationarity_tests.csv"))
show_figure("12_acf_pacf.png")
show_figure("13_time_series_decomposition.png")